# PySpark Technical Interview Questions - Data Manipulation

## 📚 Overview

This notebook covers the **most common data manipulation questions** asked in PySpark technical interviews.

**Topics Covered:**
- Finding duplicate records
- N-th highest value problems
- Running totals and cumulative sums
- Pivot tables
- Window functions (critical!)
- Self-joins
- Array operations

**Interview Frequency:** 90%+ of data engineering interviews include at least 2-3 of these patterns.

---

## 💡 Study Tips

1. **Run each cell** and understand the output
2. **Modify the code** - try different window specs, aggregations
3. **Practice explaining** - pretend you're in an interview
4. **Time yourself** - aim for 15-20 minutes per question

---

## Setup: Create Spark Session

Every PySpark interview starts with this. Know it by heart!

In [ ]:
# Import required libraries
from pyspark.sql import SparkSession  # Main entry point for DataFrame API
from pyspark.sql.functions import (  # Built-in transformation functions
    col, lit, when,  # Column operations and conditionals
    count, sum as _sum, avg, max as _max, min as _min,  # Aggregation functions
    row_number, rank, dense_rank, lag, lead, ntile,  # Window ranking functions  
    explode, split, concat, concat_ws,  # Array and string operations
    lower, upper, trim,  # String manipulation
    year, month, dayofmonth, date_format, datediff, to_date,  # Date functions
    regexp_replace, regexp_extract, substring, desc  # Regex and sorting
)
from pyspark.sql.window import Window  # For window function specifications
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType  # Schema types

# Create Spark Session
spark = SparkSession.builder \  # Builder pattern for configuration
    .appName("Interview_DataManipulation") \  # Application name (shows in Spark UI)
    .master("local[*]") \  # Run locally using all available CPU cores
    .config("spark.sql.shuffle.partitions", "4") \  # IMPORTANT: Partitions for shuffles (default=200, we use 4 for local)
    .getOrCreate()  # Create new session or get existing one

# Reduce logging verbosity  
spark.sparkContext.setLogLevel("ERROR")  # Only show ERROR messages, hide INFO/WARN

print("✅ Spark Session Created")  # Confirmation message
print(f"Spark Version: {spark.version}")  # Display current Spark version

---

# 🔀 UNDERSTANDING SHUFFLE PARTITIONS

## What Are Shuffle Partitions?

Notice this line in the code above:
```python
.config("spark.sql.shuffle.partitions", "4")
```

**Shuffle partitions** control how many partitions Spark creates when **redistributing data** across the cluster during:

| Operation | Why It Shuffles |
|-----------|----------------|
| **JOIN** | Data must be co-located by join key |
| **GROUP BY** | Data must be grouped by aggregation key |
| **Window functions** | When PARTITION BY is used |
| **DISTINCT** | Finding unique values across partitions |
| **ORDER BY** | Sorting data globally |

## 🎯 Why We Set It to 4

- **Default value**: 200 partitions ❌ (WAY too many for local development)
- **Our setting**: 4 partitions ✅ (Good for local with small data)
- **Production**: Calculate based on data size

## 📏 How to Choose the Right Number

### Golden Rule:
**Target 128-200 MB per partition**

### Formula:
```
Partitions = Data Size (MB) / 128 MB
```

### Examples:

| Data Size | Calculation | Partitions |
|-----------|-------------|------------|
| Local (< 1GB) | Small data | **2-10** |
| 10 GB | 10,000 MB / 128 MB | **~80** |
| 100 GB | 100,000 MB / 128 MB | **~800** |
| 1 TB | 1,000,000 MB / 128 MB | **~8,000** |

## ⚠️ Common Mistakes

### Mistake 1: Too Few Partitions

```python
# BAD: 100GB data with only 4 partitions
spark.conf.set("spark.sql.shuffle.partitions", "4")
large_df.groupBy("customer").sum("amount")  # Each partition = 25GB! OOM!
```

**Problems:**
- Each partition is 25GB (won't fit in memory)
- Limited parallelism (only 4 tasks)
- OutOfMemoryError

### Mistake 2: Too Many Partitions

```python
# BAD: 1GB data with 10,000 partitions  
spark.conf.set("spark.sql.shuffle.partitions", "10000")
small_df.groupBy("category").count()  # Each partition = 100KB!
```

**Problems:**
- Too many tiny tasks (scheduling overhead)
- Each partition only 100KB
- Slow performance

## 💡 Interview Q&A

### Q: "What are shuffle partitions?"

**A:** "Shuffle partitions control how many partitions Spark creates when redistributing data during operations like joins, group by, or window functions. Set via `spark.sql.shuffle.partitions`. Default is 200."

### Q: "How do you optimize shuffle partitions?"

**A:** "Calculate based on data size. Target 128-200MB per partition. For 100GB data, use ~600-800 partitions. For local development with small data, use 2-10. Enable Adaptive Query Execution (AQE) in Spark 3.0+ for automatic optimization."

### Q: "What happens if it's set wrong?"

**A:** "Too few partitions cause OOM errors and limited parallelism. Too many partitions create task scheduling overhead. Both hurt performance significantly."

## 🔍 How to Check Current Setting

```python
# Check current configuration
current = spark.conf.get("spark.sql.shuffle.partitions")
print(f"Shuffle partitions: {current}")

# Check actual partitions in DataFrame
df.rdd.getNumPartitions()  # Before shuffle
result.rdd.getNumPartitions()  # After shuffle operation
```

---

📖 **Full detailed guide**: [shuffle_partitions_explanation.md](shuffle_partitions_explanation.md)

---

---

# QUESTION 1: Find Duplicate Records

## 📝 Problem Statement

Given a dataset of employees, **find all duplicate records** based on email address.
Return the email and count of duplicates.

## 🎯 Interview Focus
- **Frequency**: Very common (80%+ of interviews)
- **Tests**: GroupBy, aggregation, filtering
- **Follow-up**: "How to find ALL duplicate rows, not just counts?"

## 💡 Key Concepts
- Use `groupBy()` + `count()` to find duplicates
- Filter where count > 1
- Join back to get full duplicate rows

In [ ]:
# Create sample employee data with duplicates
data = [
    (1, "John Doe", "john@company.com"),
    (2, "Jane Smith", "jane@company.com"),
    (3, "John Doe", "john@company.com"),     # Duplicate email
    (4, "Bob Wilson", "bob@company.com"),
    (5, "Jane Smith", "jane@company.com"),   # Duplicate email
    (6, "Alice Brown", "alice@company.com"),
]

df = spark.createDataFrame(data, ["id", "name", "email"])

print("Original Data:")
df.show()

In [ ]:
# SOLUTION 1: Find duplicate email addresses with counts

# Step 1: Group by email and count occurrences
# Step 2: Filter where count > 1 (duplicates)
# Step 3: Order by count descending to see worst offenders first

duplicates = df.groupBy("email") \
    .agg(count("*").alias("count")) \
    .filter(col("count") > 1) \
    .orderBy(col("count").desc())

print("\n📊 Duplicate Email Addresses:")
duplicates.show()

# Interview Tip: Always explain your aggregation logic clearly!

In [ ]:
# SOLUTION 2: Get ALL rows with duplicate emails (common follow-up question)

# Step 1: Get just the duplicate emails
duplicate_emails = duplicates.select("email")

# Step 2: Join back to original data to get all duplicate rows
# Using INNER join keeps only matching rows
all_duplicates = df.join(duplicate_emails, "email", "inner")

print("\n📋 All Rows with Duplicate Emails:")
all_duplicates.show()

# Interview Tip: Mention that this is a "semi-join" pattern
# Alternative: df.join(duplicate_emails, "email", "left_semi")

### ✅ Key Takeaways - Question 1

1. **Pattern**: `groupBy` + `agg` + `filter` for finding duplicates
2. **Performance**: This causes a shuffle - consider partitioning by the duplicate column if data is large
3. **Variations**: Can be asked for phone numbers, SSNs, IP addresses, etc.

---

# QUESTION 2: Second Highest Salary

## 📝 Problem Statement

Find the **second highest salary** from employees table.
If there's no second highest, return NULL.

## 🎯 Interview Focus
- **Frequency**: Extremely common (95%+ of interviews)
- **Tests**: Window functions, ranking, handling ties
- **Follow-up**: "What about Nth highest?" or "What if there are ties?"

## 💡 Key Concepts
- Use `dense_rank()` to handle ties properly
- `rank()` vs `row_number()` vs `dense_rank()` - know the difference!
- Always consider edge cases (less than 2 distinct salaries)

In [ ]:
# Create employee salary data
data = [
    ("John", "Engineering", 95000),
    ("Jane", "Engineering", 88000),
    ("Bob", "Sales", 75000),
    ("Alice", "Engineering", 95000),  # Tied for highest!
    ("Charlie", "Sales", 82000),
]

df = spark.createDataFrame(data, ["name", "department", "salary"])

print("Employee Salary Data:")
df.show()

In [ ]:
# SOLUTION 1: Using dense_rank() - Handles ties correctly

# Step 1: Create window spec - order by salary descending
# No partition means it's a global ranking
window_spec = Window.orderBy(col("salary").desc())

# Step 2: Get distinct salaries (important for ties!)
# Step 3: Rank them using dense_rank
# Step 4: Filter for rank = 2

result = df.select("salary").distinct() \
    .withColumn("rank", dense_rank().over(window_spec)) \
    .filter(col("rank") == 2) \
    .select("salary")

print("\n💰 Second Highest Salary (using dense_rank):")
result.show()

# Interview Tip: Explain why dense_rank is better than rank or row_number for this problem

In [ ]:
# SOLUTION 2: Alternative approach - Using offset/collect
# Simpler but less flexible for "Nth highest"

distinct_salaries = df.select("salary").distinct().orderBy(col("salary").desc())

# Check if we have at least 2 distinct salaries
if distinct_salaries.count() >= 2:
    second_highest = distinct_salaries.collect()[1][0]
    print(f"\n💰 Second Highest Salary (using collect): ${second_highest:,}")
else:
    print("\n⚠️ Second Highest Salary: NULL (not enough distinct salaries)")

# Interview Warning: collect() brings data to driver - only use for small results!

In [ ]:
# BONUS: Understanding rank() vs row_number() vs dense_rank()

print("\n📊 Comparison of Ranking Functions:")
print("\nSalaries: [95000, 95000, 88000, 82000, 75000]")
print("\nrow_number():  1, 2, 3, 4, 5  (Always sequential, no ties)")
print("rank():        1, 1, 3, 4, 5  (Ties allowed, skips ranks)")
print("dense_rank():  1, 1, 2, 3, 4  (Ties allowed, no skipping)")

# Show actual example
window_spec = Window.orderBy(col("salary").desc())

result = df.withColumn("row_num", row_number().over(window_spec)) \
    .withColumn("rank", rank().over(window_spec)) \
    .withColumn("dense_rank", dense_rank().over(window_spec)) \
    .orderBy("rank")

print("\nLive Example:")
result.show()

### ✅ Key Takeaways - Question 2

1. **Always use `dense_rank()`** for "Nth highest" problems to handle ties
2. **Remember to use `.distinct()`** before ranking
3. **Handle edge case**: What if N > total distinct values?
4. **Performance**: Window functions cause shuffle - unavoidable here

---

# QUESTION 3: Running Total (Cumulative Sum)

## 📝 Problem Statement

Calculate the **running total** of sales for each product over time.

## 🎯 Interview Focus
- **Frequency**: Very common (70%+ of interviews)
- **Tests**: Window functions with frames, cumulative calculations
- **Follow-up**: "What about moving average?" or "How to calculate YTD totals?"

## 💡 Key Concepts
- Window frames: `rowsBetween` vs `rangeBetween`
- `unboundedPreceding` to `currentRow` for running totals
- Partition by group, order by time

In [ ]:
# Create daily sales data for multiple products
data = [
    ("Product_A", "2024-01-01", 100),
    ("Product_A", "2024-01-02", 150),
    ("Product_A", "2024-01-03", 200),
    ("Product_B", "2024-01-01", 80),
    ("Product_B", "2024-01-02", 120),
    ("Product_B", "2024-01-03", 90),
]

df = spark.createDataFrame(data, ["product", "date", "sales"])

print("Daily Sales Data:")
df.show()

In [ ]:
# SOLUTION: Running Total using Window Function

# Step 1: Define window specification
# - Partition by product (separate running total for each product)
# - Order by date (chronological order)
# - Frame: from beginning to current row (unboundedPreceding to currentRow)

window_spec = Window.partitionBy("product") \
    .orderBy("date") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

# Step 2: Apply sum aggregation over the window
result = df.withColumn("running_total", _sum("sales").over(window_spec))

print("\n📈 Running Total by Product:")
result.show()

# Interview Tip: Explain that rowsBetween defines the "frame" of rows to aggregate

In [ ]:
# BONUS: Different Window Frame Specifications

print("\n📚 Understanding Window Frames:\n")

# 1. Running total (from start to current)
window_running = Window.partitionBy("product").orderBy("date") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

# 2. 3-day moving average (current + 2 previous days)
window_3day = Window.partitionBy("product").orderBy("date") \
    .rowsBetween(-2, Window.currentRow)

# 3. Forward-looking (current + next 2 days)
window_forward = Window.partitionBy("product").orderBy("date") \
    .rowsBetween(Window.currentRow, 2)

result = df.withColumn("running_total", _sum("sales").over(window_running)) \
    .withColumn("moving_avg_3day", avg("sales").over(window_3day)) \
    .withColumn("forward_sum", _sum("sales").over(window_forward))

print("Different Window Calculations:")
result.show()

print("\n💡 Frame Types:")
print("- rowsBetween(-2, 0): Last 3 rows including current")
print("- rowsBetween(unboundedPreceding, 0): All previous + current")
print("- rowsBetween(0, 2): Current + next 2 rows")

### ✅ Key Takeaways - Question 3

1. **Pattern**: `Window.partitionBy().orderBy().rowsBetween()`
2. **Running Total**: Use `unboundedPreceding` to `currentRow`
3. **Moving Average**: Use fixed window like `rowsBetween(-N, 0)`
4. **Performance**: Window functions require sorting within partitions

---

# Continue practicing...

The remaining questions follow the same pattern:
1. Problem statement with context
2. Sample data creation
3. Step-by-step solution with comments
4. Key takeaways

**Practice Questions 4-10:**
- Q4: Pivot Tables
- Q5: Remove Consecutive Duplicates  
- Q6: Top N Per Group (very important!)
- Q7: Self-Join (Manager Hierarchy)
- Q8: Explode Nested Arrays
- Q9: Moving Average
- Q10: Find Missing Dates

💡 **Interview Tip**: In a real interview, you'd typically solve 1-2 questions in 45-60 minutes.